# Lab 10 -- Decision Trees

Compared to last week, this is a very simple lab <span style="font-size:20pt;">😃</span> You'll have fun programming!

You will implement the **Classification and Regression Tree (CART)** algorithm from scratch.

The lab is broken down into the following pieces:

* Regression Criterion
* Creating Splits
* Buiding a Tree
* Making a prediction


# Decision trees for Regression
## Exercise 1 -- Download and load the dataset

We will be using the usual Boston Housing dataset, which is available to download from ECLASS

* Download the file
* Read it and separate the target variable from the features.
* Make a 80/10/10 train/validation/test split

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [14]:
housing_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
data = pd.read_table("housing.txt", names=housing_names, sep=r'\s+')

The target variable will be as usual `MEDV`. Use the rest as features.

In [15]:
X = data.iloc[:, :-1].values
y = data["MEDV"].values


X_train, X_aux, y_train, y_aux = train_test_split(X, y, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X_aux, y_aux, test_size=0.5)

## Exercise 2 -- Optimization Criterion

For regression, a simple criterion to optimize is to minimize the sum of squared errors for a given region. This is, for all datapoints in a region with size, we minimize:

$$\sum_{i=1}^N(y_i - \hat{y})^2$$

where $N$ is the number of datapoits in the region and $\hat{y}$ is the mean value of the region for the target variable. 

Implement such a function using the description below.

Please, don't use an existing implementation, refer to the [book](https://www.statlearning.com/s/ISLRSeventhPrinting.pdf), and if you need help, ask questions!

In [16]:
def regression_criterion(region: np.ndarray):
    """
    Implements the sum of squared error criterion in a region
    
    Parameters
    ----------
    region : ndarray
        Array of shape (N,) containing the values of the target values 
        for N datapoints in the training set.
    
    Returns
    -------
    float
        The sum of squared error
        
    Note
    ----
    The error for an empty region should be infinity (use: float("inf"))
    This avoids creating empty regions
    """
    if len(region) == 0:
        return float("inf")
    
    return np.sum((region - np.mean(region))**2)

In [17]:
# test your code
rng = np.random.default_rng(0)
print(regression_criterion(rng.random(size=40)))
print(regression_criterion(np.ones(10)))
print(regression_criterion(np.zeros(10)))
print(regression_criterion(np.array([])))

3.6200679838629544
0.0
0.0
inf


## Exercise 3 -- Make a split

In [18]:
def split_region(region: np.ndarray, feature_index: int, tau: float):
    """
    Given a region, splits it based on the feature indicated by
    `feature_index`, the region will be split in two, where
    one side will contain all points with the feature with values 
    lower than `tau`, and the other split will contain the 
    remaining datapoints.
    
    Parameters
    ----------
    region : array of size (n_samples, n_features)
        a partition of the dataset (or the full dataset) to be split
    feature_index : int
        the index of the feature (column of the region array) used to make this partition
    tau : float
        The threshold used to make this partition
        
    Return
    ------
    left_partition : array
        indices of the datapoints in `region` where feature < `tau`
    right_partition : array
        indices of the datapoints in `region` where feature >= `tau` 
    """
    
    left_partition = region[:,feature_index] < tau
    right_partition = ~left_partition

    return left_partition, right_partition

## Exercise 4 -- Find the best split

The strategy is quite simple (as well as inefficient), but it helps to reinforce the concepts.
We are going to use a greedy, exhaustive algorithm to select splits, selecting the `feature_index` and the `tau` that minimizes the Regression Criterion

In [19]:
def get_split(X: np.ndarray, y:np.ndarray) -> dict[str, float | np.ndarray]:
    """
    Given a dataset (full or partial), splits it on the feature of that minimizes the sum of squared error
    
    Parameters
    ----------
    X : array (n_samples, n_features)
        features 
    y : array (n_samples, )
        labels
    
    Returns
    -------
    node : dictionary
        keys are:
        * 'feature_index' -> an integer that indicates the feature (column) of `X` on which the data is split
        * 'tau' -> the threshold used to make the split
        * 'left_region' -> array of indices where the `feature_index`th feature of X is lower than `tau`
        * 'right_region' -> indices not in `low_region`
    """
    best_sse = float("inf")

    best_tau = None
    best_feature = None

    # devemos ir por todas as features
    for each_feature in range(X.shape[1]):
        mask = X[:, each_feature].argsort() # retorna indices em ordem crescente

        # Acho o tau depois de achar o tau eu posso achar os indices das regioes
        X_sorted= X[mask, each_feature]
        y_sorted = y[mask] 

        for each_idy in range(1, y.shape[0]):
            
            # evitar splits impossiveis
            if X_sorted[each_idy] == X_sorted[each_idy -1]:
                continue

            sse = (
                regression_criterion(y_sorted[:each_idy])
                + regression_criterion(y_sorted[each_idy:])
            )

            if sse < best_sse:
                best_sse = sse
                best_feature = each_feature
                best_tau = (X_sorted[each_idy] + X_sorted[each_idy - 1]) / 2

    left_partition, right_partition =  split_region(X, best_feature, best_tau)

    node = {
        "feature_index":best_feature,
        "tau":best_tau,
        "left_region":left_partition,
        "right_region":right_partition
    }

    return node

## Exercise 5 -- Recursive Splitting

The test above is an example on how to find the root node of our decision tree. The algorithm now is a greedy search until we reach a stop criterion. To find the actual root node of our decision tree, you must provide the whole training set, not just a slice of 15 rows as the test above.

The trivial stopping criterion is to recursively grow the tree until each split contains a single point (perfect node purity). If we go that far, it normally means we are overfitting.

You will implement these criteria to stop the growth:

* A node is a leaf if:
    * It has less than `min_samples` datapoints
    * It is at the `max_depth` level from the root (each split creates a new level)
    * The criterion is `0`



In [20]:
def recursive_growth(min_samples, max_depth, current_depth, X, y):
    """
    Recursively grows a decision tree.
    
    Parameters
    ----------

    min_samples : int
        parameter for stopping criterion if a node has <= min_samples datapoints
    max_depth : int
        parameter for stopping criterion if a node belongs to this depth
    depth : int
        current distance from the root
    X : array (n_samples, n_features)
        features (full dataset)
    y : array (n_samples, )
        labels (full dataset)

    returns
    ------
    node : dictionary
        If the node is terminal, it contains only the "value" key, which determines the value to be used as a prediction.
        If the node is not terminal, the dictionary has the structure defined by `get_split`
    
    Notes
    -----
    To create a terminal node, a dictionary is created with a single "value" key, with a value that
    is the mean of the target variable
    
    'left' and 'right' keys are added to non-terminal nodes, which contain (possibly terminal) nodes 
    from higher levels of the tree:
    'left' corresponds to the 'left_region' key, and 'right' to the 'right_region' key
    """

    is_leaf = len(y) < min_samples or current_depth >= max_depth or regression_criterion(y) == 0.0

    if is_leaf:
        return {"value": y.mean()}
   

    node = get_split(X, y)

    node["left"] = recursive_growth(
        min_samples, max_depth, current_depth + 1,
        X[node["left_region"]],
        y[node["left_region"]]
    )

    node["right"] = recursive_growth(
        min_samples, max_depth, current_depth + 1,
        X[node["right_region"]],
        y[node["right_region"]]
    )

    return node

Below we provide code to visualise the generated tree!

In [21]:
def print_tree(root, depth):
    pass


In [22]:
#print_tree(root, 0)

Resultado esperado: <br>

```text
 X_4 < 0.581
.   X_10 < 15.9
.  .   X_1 < 20.0
.  .  .   X_2 < 25.65
.  .  .  .   X_6 < 76.7
.  .  .  .   [24.38888888888889]
.  .  .  .   [24.38888888888889]
.  .  .   [21.852173913043476]
.  .   [20.616666666666664]
.  .   X_0 < 0.01439
.  .   [28.200000000000003]
.  .  .   X_0 < 0.0456
.  .  .  .   X_6 < 92.9
.  .  .  .  .   X_0 < 0.04527
.  .  .  .  .   [23.0]
.  .  .  .  .  .   X_7 < 1.5331
.  .  .  .  .  .   [17.2]
.  .  .  .  .  .   [17.2]
.  .  .  .   [22.284000000000002]
.  .  .  .   X_9 < 226.0
.  .  .  .   [18.631249999999998]
.  .  .  .  .   X_4 < 0.413
.  .  .  .  .   [24.09090909090909]
.  .  .  .  .  .   X_5 < 8.337
.  .  .  .  .  .   [22.05960264900662]
.  .  .  .  .  .   [22.05960264900662]
.   X_7 < 2.5671
.  .   X_0 < 9.82349
.  .  .   X_7 < 10.7103
.  .  .  .   X_5 < 5.403
.  .  .  .   [13.95]
.  .  .  .  .   X_4 < 0.439
.  .  .  .  .  .   X_0 < 3.69695
.  .  .  .  .  .   [23.247058823529407]
.  .  .  .  .  .   [23.247058823529407]
.  .  .  .  .  .   X_4 < 0.493
.  .  .  .  .  .   [19.64375]
.  .  .  .  .  .   [19.64375]
.  .  .   [21.735156249999996]
.  .   [22.175193798449612]
.   [22.32945205479452]
```

## Exercise 6 -- Make a Prediction
Use the a node to predict the class of a compatible dataset

In [23]:
def predict_sample(node, sample:np.ndarray):
    """
    Makes a prediction based on the decision tree defined by `node`
    
    Parameters
    ----------
    node : dictionary
        A node created one of the methods above
    sample : array of size (n_features,)
        a sample datapoint

    returns
    -------
    float: predicted value

    """
    is_leaf = "value" in node
    if is_leaf:
        return node["value"]
    
    current_feature = node["feature_index"]
    current_tau = node["tau"]
    left = node["left"]
    right = node["right"]

    if sample[current_feature] < current_tau:
        return predict_sample(left, sample)
    else:
        return predict_sample(right, sample)

        
def predict(node, X):
    """
    Makes a prediction based on the decision tree defined by `node`
    
    Parameters
    ----------
    node : dictionary
        A node created one of the methods above
    X : array of size (n_samples, n_features)
        n_samples predictions will be made
    """
    size = X.shape[0]
    y = np.zeros(size)

    for i in range(size):
        y[i] = predict_sample(node, X[i])

    return y

Now use the functions defined above to calculate the RMSE of the validation set. 
* Try first with `min_samples=20` and `max_depth=6` (for this values you should get a validation RMSE of ~8.8)

Then, experiment with different values for the stopping criteria.

In [24]:
# calculate root mean squared error with numpy
def root_mean_squared_error(y_true, y_pred):
    """
    Calculates the root mean squared error between two arrays
    
    Parameters
    ----------
    y_true : array of size (n_samples,)
        true labels
    y_pred : array of size (n_samples,)
        predicted labels
    """
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [25]:
min_samples = 20
max_depth = 6
root = recursive_growth(min_samples, max_depth, 0, X_train, y_train)
train_rmse = root_mean_squared_error(y_train, predict(root, X_train))
test_rmse = root_mean_squared_error(y_test, predict(root, X_test))

print(f'Train RMSE : {train_rmse}')
print(f'Test RMSE : {test_rmse}')

Train RMSE : 2.467170622531166
Test RMSE : 3.2217994267532752


```text
Train MSE : 9.624284478888335
Test MSE : 9.148171905205656
```

## Just to see how much we can improve this naive decision tree!

In [26]:
from sklearn.linear_model import LinearRegression

In [27]:
reg = LinearRegression().fit(X_train, y_train)

In [28]:
reg_train_rmse = root_mean_squared_error(y_train, reg.predict(X_train))
reg_test_rmse = root_mean_squared_error(y_test, reg.predict(X_test))
print(f'Regression Train MSE : {reg_train_rmse}')
print(f'Regression Test MSE : {reg_test_rmse}')

Regression Train MSE : 4.724812353964074
Regression Test MSE : 4.9738803339207625


esperado:
```text
Regression Train MSE : 4.396188144698282
Regression Test MSE : 5.931426809490862
```

# Decision trees for Classification
You will implement decision trees for classification using the Gini index as the splitting criterion. You’ll build the tree recursively, selecting splits that minimize Gini impurity and classifying samples based on majority class in each leaf. A good dataset to start with is the Iris dataset, which is small, well-labeled, and available directly via `sklearn.datasets.load_iris().`

In [29]:
from sklearn.datasets import load_iris

# Load the Iris dataset
data = load_iris()
X = data.data
y = data.target

We will focus only on binary classification today!

In [30]:
X = X[y != 2]
y = y[y != 2]

In [31]:
# split your data into training, validation and test sets!
X_train, X_aux, y_train, y_aux = train_test_split(X, y, test_size=0.3)
X_val, X_test, y_val, y_test = train_test_split(X_aux, y_aux, test_size=0.5)

### Feel free to use the same code as for regression, but change the criterion and the prediction function. You can also implement a new one if you want to!

In [ ]:
# you will test 3 values for min_samples_split: 2, 4, 6
# Remember that this sets the minimum number of samples required in a node to be eligible for splitting. 
# These values are good for small datasets like Iris, but you can try other values for larger datasets to not make the tree too deep.

# your code goes here
class Node:
    def __init__(self, feature, tau, left_partition, right_partition):
        self.feature = feature
        self.tau = tau
        self.left_region = left_partition
        self.right_region = right_partition
        self.left = None
        self.right = None

class Leaf:
    def __init__(self, value):
        self.value = value

class Tree:
    def __init__(self):
        self.root = None

    def classification_criterion(self, region: np.ndarray):
        if len(region) == 0:
            return float("inf")

        x = np.mean(region)
        
        return 2 * x * (1 - x)
    
    def split_region(self, region: np.ndarray, idfeature: int, tau:float):
        left_partition = region[:,idfeature] < tau
        right_partition = ~left_partition

        return left_partition, right_partition
    
    def get_split(self, X: np.ndarray, y:np.ndarray) -> dict[str, float | np.ndarray]:
        best_gini = float("inf")

        best_tau = None
        best_feature = None

        N = y.shape[0]

        # devemos ir por todas as features
        for each_feature in range(X.shape[1]):
            mask = X[:, each_feature].argsort() # retorna indices em ordem crescente

            # Acho o tau depois de achar o tau eu posso achar os indices das regioes
            X_sorted= X[mask, each_feature]
            y_sorted = y[mask] 

            for each_idy in range(1, y.shape[0]):
                
                # evitar splits impossiveis
                if X_sorted[each_idy] == X_sorted[each_idy -1]:
                    continue

                left = y_sorted[:each_idy]
                right = y_sorted[each_idy:]

                gini = (
                    (len(left) / N) * self.classification_criterion(left)
                    + (len(right) / N) * self.classification_criterion(right)
                )

                if gini < best_gini:
                    best_gini = gini
                    best_feature = each_feature
                    best_tau = (X_sorted[each_idy] + X_sorted[each_idy - 1]) / 2

        left_partition, right_partition =  self.split_region(X, best_feature, best_tau)

        node = Node(
            best_feature,
            best_tau,
            left_partition,
            right_partition
        )

        return node
    
    
    def mode(self, y: np.ndarray):

        values, counts = np.unique(y, return_counts=True)

        id_mode = counts.argmax()
        mode = values[id_mode]

        return mode


    def recursive_growth(self, min_samples, max_depth, current_depth, X, y):

        is_leaf = len(y) < min_samples or current_depth >= max_depth or self.classification_criterion(y) == 0.0

        if is_leaf:
            return Leaf(self.mode(y))
    

        node = self.get_split(X, y)

        node.left = self.recursive_growth(
            min_samples, max_depth, current_depth + 1,
            X[node.left_region],
            y[node.left_region]
        )

        node.right = self.recursive_growth(
            min_samples, max_depth, current_depth + 1,
            X[node.right_region],
            y[node.right_region]
        )

        return node
    
    def predict_sample(self, node, sample: np.ndarray):
        is_leaf = isinstance(node, Leaf)
        if is_leaf:
            return node.value
        
        if sample[node.feature] < node.tau:
            return self.predict_sample(node.left, sample)
        else: 
            return self.predict_sample(node.right, sample)
    
    def predict(self, X):
        size = X.shape[0]
        y = np.zeros(size)

        for i in range(size):
            y[i] = self.predict_sample(self.root, X[i])

        return y

    def fit(self, X, y, min_samples, max_depth):
        self.root = self.recursive_growth(
            min_samples,
            max_depth,
            0,
            X,
            y
        )

In [33]:
# you will test 3 values for min_samples_split: 2, 4, 6
# Remember that this sets the minimum number of samples required in a node to be eligible for splitting. 
# These values are good for small datasets like Iris, but you can try other values for larger datasets to not make the tree too deep.

tree = Tree()
tree.fit(X_train, y_train, 2, 6)
y_pred = tree.predict(X_test)

Use accuracy to find the best split. Don't import it from sklearn, calculate it yourself, it's a one-liner ;)

In [34]:
## Lastly, evaluate your model on the test set and print the accuracy score

def accuracy_score(y_true, y_pred):
    return np.mean(y_true == y_pred)

accuracy_score(y_test, y_pred)

np.float64(1.0)